In [1]:
import json

In [2]:
mapping = json.load(open('/home/fellah-a/Documents/Text2KG/data/knowledge_graphs/industry_of_future_v2/llm_based_entities_cleaning/_mappings/mapping_0.json', 'r'))

In [5]:
len(set(mapping.keys()))

118731

In [4]:
len(set(mapping.values()))

87705

In [3]:
len(set(mapping.keys())) - len(set(mapping.values()))

31026

In [12]:
from typing import TypeVar, Type, Callable, List
from returns.result import Result, safe

In [ ]:
# Type[Callable[...,int]]

NameError: name 'Type' is not defined

In [1]:
from torch_geometric.data import HeteroData
import torch
from torch_sparse import SparseTensor
import sys
sys.path.append('..')
from src.commands.feature_extraction.main import Graph

In [2]:
from typing import Callable, Dict, Tuple, Type
import torch
from torch import Tensor
from torch_geometric.data import HeteroData

EdgeType = Tuple[str, str, str]
Generator = Callable[[HeteroData], Dict[str, Tensor]]  # returns {'edge_index': ..., 'edge_attr': ...}


def _parse_key(key) -> EdgeType:

    if isinstance(key, str):
        key = tuple(key.split("__"))
    if not (isinstance(key, tuple) and len(key) == 3):
        raise ValueError(f"Invalid edge type {key!r}")
    return key


def make_lazy_heterodata(
    relations: Dict[EdgeType, Generator] = None,
    name: str = "LazyHeteroData", 
    **kwargs: Generator
) -> Type[HeteroData]:
    
    generators = {
        **relations,
        **kwargs
    }

    def __getitem__(self, *args):

        key = args[0] if len(args) == 1 else tuple(args)

        if isinstance(key, tuple) and key in generators:
            self._materialize(key)
        
        return HeteroData.__getitem__(self, *args)

    def _materialize(self, edge_type):

        # Already generated -> served from the store, never recomputed
        if edge_type in self._edge_store_dict and "edge_index" in self._edge_store_dict[edge_type]:
            return
        
        building = self.__dict__.setdefault("_lazy_building", set())

        if edge_type in building:
            raise RecursionError(f"Cyclic lazy relation at {edge_type}")
        
        building.add(edge_type)

        attrs = generators[edge_type](self)

        store = HeteroData.__getitem__(self, edge_type)

        for attr, value in attrs.items():
            store[attr] = value

    def materialize(self, *edge_types):
        """Generate all (or given) lazy relations, e.g. before to_hetero / metadata()."""
        for et in (edge_types or generators):
            self._materialize(_parse_key(et))
        return self

    return type(name, (HeteroData,), {
        "__getitem__": __getitem__,
        "_materialize": _materialize,
        "materialize": materialize,
    })


# ---------------- Use case ----------------

def _num_nodes(data, nt):
    store = HeteroData.__getitem__(data, nt)
    return store.node_id.size(0) if "node_id" in store else store.num_nodes


def _triple_lookup(data, rel, dst):
    """For each Triple, index of its dst node via rel (-1 if missing)."""
    ei = data["Triple", rel, dst].edge_index
    out = torch.full((_num_nodes(data, "Triple"),), -1, dtype=torch.long, device=ei.device)
    out[ei[0]] = ei[1]
    return out


def entity_relation_entity(data):

    head = _triple_lookup(data, "HAS-HEAD", "Entity")
    tail = _triple_lookup(data, "HAS-TAIL", "Entity")
    pred = _triple_lookup(data, "HAS-PREDICATE", "Predicate")

    valid = (head >= 0) & (tail >= 0) & (pred >= 0)

    return {
        "edge_index": torch.stack([head[valid], tail[valid]], dim=0),
        "edge_attr": data["Predicate"].x[pred[valid]],
    }


def entity_appears_in_article(data):

    n_ent, n_tri, n_art = (_num_nodes(data, t) for t in ("Entity", "Triple", "Article"))
    h = data["Triple", "HAS-HEAD", "Entity"].edge_index
    t = data["Triple", "HAS-TAIL", "Entity"].edge_index
    b = data["Triple", "BELONGSTO", "Article"].edge_index

    et = torch.cat([h, t], dim=1).flip(0)  # Entity -> Triple
    E2T = torch.sparse_coo_tensor(et, torch.ones(et.size(1)), (n_ent, n_tri)).coalesce()
    T2A = torch.sparse_coo_tensor(b, torch.ones(b.size(1)), (n_tri, n_art)).coalesce()
    E2A = torch.sparse.mm(E2T, T2A).coalesce()
    return {"edge_index": E2A.indices()}


KGData = make_lazy_heterodata({
    ("Entity", "HASRELATIONWITH", "Entity"): entity_relation_entity,
    ("Entity", "APPEARSIN", "Article"): entity_appears_in_article,
}, name="KGData")

In [3]:
G = Graph.load("/home/fellah-a/Documents/Text2KG/data/features/industry_of_future")

In [1]:
import torch

In [3]:
a = torch.rand(10)
b = torch.rand_like(a)

a, b

(tensor([0.6022, 0.9675, 0.8352, 0.7861, 0.2458, 0.7523, 0.2763, 0.7734, 0.3947,
         0.7114]),
 tensor([0.9835, 0.0800, 0.5043, 0.9403, 0.9873, 0.6955, 0.0109, 0.1410, 0.5420,
         0.2549]))

In [6]:
a + b - torch.sqrt(a * b)

tensor([0.8161, 0.7693, 0.6905, 0.8667, 0.7405, 0.7245, 0.2324, 0.5842, 0.4742,
        0.5405])

In [1]:
import torch

In [40]:
l1, l2 = 8, 16
indices = torch.randint(low=0, high=l1, size=(4,))

In [41]:
indices

tensor([4, 1, 5, 5])

In [55]:
i = torch.arange(l1).repeat(l2).reshape((l1, l2)).t().flatten()
i

tensor([0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2,
        3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5,
        6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3,
        4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6,
        7, 7, 7, 7, 7, 7, 7, 7])

In [54]:
torch.eye(l1)[i].shape

torch.Size([128, 8])

In [44]:
from functools import reduce

In [50]:
[1, 2, 3, 4, 5, 6][:-2]

[1, 2, 3, 4]

In [ ]:
reduce(lambda a, b : a * b, [1, 2, 3], i)

6

In [3]:
import os

for key in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY",
            "http_proxy", "https_proxy", "all_proxy"):
    if os.environ.get(key):
        print(f"{key} is configured")

In [1]:
from neo4j import GraphDatabase

password = "j6NeVnQZqgHekniGWYg0wOFRo7zA4mNvHjagYZ36smg"

driver = GraphDatabase.driver(
    "neo4j+s://1387e3cc.databases.neo4j.io",
    auth=("neo4j", password),
)

In [2]:
driver.verify_connectivity()

Unable to retrieve routing information


ServiceUnavailable: Unable to retrieve routing information

In [1]:
import torch

In [2]:
D = torch.load('/home/fellah-a/Documents/Text2KG/experiments/topic_modeling/pooling_documents_reconstruction/results/predictions_DMoN_restore_norm=true;norm_pool=true.pt')

In [3]:
D

{'xd_hat': tensor([[ 0.5319, -0.3545, -0.2029,  ..., -0.2822,  0.1998, -0.2423],
         [ 0.1323, -0.1844, -0.0930,  ..., -0.2434,  0.1198, -0.1136],
         [ 0.3478, -0.2575, -0.1328,  ..., -0.2644,  0.1374, -0.1619],
         ...,
         [ 0.4684, -0.2513, -0.1231,  ..., -0.2104,  0.1015, -0.1600],
         [ 0.6710, -0.3735, -0.1676,  ..., -0.2681,  0.1517, -0.2118],
         [ 0.3280, -0.2674, -0.1542,  ..., -0.2506,  0.1670, -0.1624]]),
 'x_hidden': tensor([[ 0.0644,  0.1394, -0.2102,  ..., -0.0060,  0.1262, -0.0363],
         [ 0.1298,  0.0711, -0.1767,  ...,  0.0024,  0.1368, -0.0301],
         [ 0.0977,  0.1593, -0.1930,  ...,  0.0067,  0.1281, -0.0566],
         ...,
         [ 0.1193,  0.0902, -0.1825,  ...,  0.0023,  0.1349, -0.0346],
         [ 0.0686,  0.1127, -0.2005,  ..., -0.0075,  0.1234, -0.0280],
         [ 0.0845,  0.1134, -0.1722,  ...,  0.0009,  0.1121, -0.0385]]),
 'x_hat': tensor([[ 0.6629, -0.4398, -0.2266,  ..., -0.3453,  0.1949, -0.3121],
         [ 0.8

In [1]:
from pydantic import BaseModel

In [10]:
class Address(BaseModel):
    host : str
    port : int

    @classmethod
    def parse(cls, addr : str):
        host, port = addr.split(':')
        port = int(port)
        return cls(host=host, port=port)

In [12]:
from typing import Literal

In [16]:
T = Literal['1', '2']

In [ ]:
T.

typing.Literal['1', '2']

In [11]:
Address.parse("localhost:8000")

Address(host='localhost', port=8000)

In [23]:
import requests
import json

In [34]:
response = requests.post("http://0.0.0.0:8000/corpuses", json={
    "id" : "test",
    "config" : {
        "from_" : "*",
        "to" : "*",
        "teams" : { "hal" : ["ls2n"] },
        "page_size" : 32
    }
})

In [2]:
import httpx

In [14]:
corpus_id = "test"
url = "http://localhost:8000/jobs/download/{corpus_id}/start".format(corpus_id=corpus_id)

response = httpx.post(url=url)
print(response.status_code)
print(response.json())
job_id = response.json().get('id')

200
{'id': 'e35324c5-2a6f-42e0-be0e-eabbd7a25ec7', 'status': 'PENDING', 'type': 'DOWNLOAD'}


In [15]:
sse_url = "http://localhost:8000/jobs/sse/{job_id}".format(job_id=job_id)

with httpx.stream("GET", sse_url, timeout=None) as response:

    response.raise_for_status()

    for line in response.iter_lines():
        if line:
            print(line)

data: {"task":"main","progress":0.0,"timestamp":"2026-10-28 14:27:21","level":"INFO","message":"A total of 31 pages to download.","extra":{"num_processed":0,"total_skipped":0,"total_saved":0}}
data: {"task":"main","progress":0.03225806451612903,"timestamp":"2026-10-28 14:27:23","level":"INFO","message":"Page 0 downloaded with 32 documents.","extra":{"num_processed":1,"total_skipped":0,"total_saved":0}}
data: {"task":"main","progress":0.03225806451612903,"timestamp":"2026-10-28 14:27:23","level":"INFO","message":"Page 0 with 32 documents was saved to disk.","extra":{"num_processed":1,"total_skipped":0,"total_saved":1}}
data: {"task":"main","progress":0.06451612903225806,"timestamp":"2026-10-28 14:27:26","level":"INFO","message":"Page 1 downloaded with 32 documents.","extra":{"num_processed":2,"total_skipped":0,"total_saved":1}}
data: {"task":"main","progress":0.06451612903225806,"timestamp":"2026-10-28 14:27:26","level":"INFO","message":"Page 1 with 32 documents was saved to disk.","ext

KeyboardInterrupt: 

In [16]:
import json

In [23]:
template = """{{
    "corpus_id" : "{corpus_id}",
    "glob" : {{
        "domain" : "industry_of_future",
        "batch_size" : 1,
        "buffer_size" : 4,
        "resume" : true
    }},
    "stages" : {{
        "corpus_cleaning" : {{
            "stage" : "corpus_cleaning",
            "batch_size" : 32,
            "buffer_size" : 32
        }},
        "coreference_resolution" : {{
            "stage" : "coreference_resolution",
            "model" : "spanbert-large",
            "device" : -1
        }},
        "keywords_extraction" : {{
            "stage" : "keywords_extraction",
            "batch_size" : 1,
            "buffer_size" : 4
        }},
        "sentence_segmentation" : {{
            "stage" : "sentence_segmentation",
            "use_gpu" : false,
            "batch_size" : 128,
            "buffer_size" : 128
        }},
        "acronyms_extraction" : {{
            "stage" : "acronyms_extraction",
            "qa_model" : "timpal0l/mdeberta-v3-base-squad2",
            "threshold" : 0.5
        }},
        "triples_extraction": {{
            "stage" : "triples_extraction",
            "batch_size" : 16,
            "buffer_size" : 16,
            "type" : "two-step-llm/{llm_name}"
        }},
        "entities_normalization" : {{
            "stage" : "entities_normalization",
            "batch_size" : 128,
            "buffer_size" : 128
        }},
        "rule_based_predicate_cleaning" : {{
            "stage" : "rule_based_predicate_cleaning",
            "batch_size" : 128,
            "buffer_size" : 128
        }},
        "llm_based_predicate_cleaning" : {{
            "llm_name" : "{llm_name}",
            "stage" : "llm_based_predicate_cleaning",
            "batch_size" : 128,
            "buffer_size" : 128
        }},
        "llm_based_entities_cleaning" : {{
            "llm_name" : "{llm_name}",
            "stage" : "llm_based_entities_cleaning",
            "batch_size" : 128,
            "buffer_size" : 128,
            "plm_device" : -1
        }}
    }}
}}"""

In [24]:
config = template.format(corpus_id="test/data.jsonl", llm_name="deepseek-v4-flash")

In [26]:
json.loads(config)

{'corpus_id': 'test/data.jsonl',
 'glob': {'domain': 'industry_of_future',
  'batch_size': 1,
  'buffer_size': 4,
  'resume': True},
 'stages': {'corpus_cleaning': {'stage': 'corpus_cleaning',
   'batch_size': 32,
   'buffer_size': 32},
  'coreference_resolution': {'stage': 'coreference_resolution',
   'model': 'spanbert-large',
   'device': -1},
  'keywords_extraction': {'stage': 'keywords_extraction',
   'batch_size': 1,
   'buffer_size': 4},
  'sentence_segmentation': {'stage': 'sentence_segmentation',
   'use_gpu': False,
   'batch_size': 128,
   'buffer_size': 128},
  'acronyms_extraction': {'stage': 'acronyms_extraction',
   'qa_model': 'timpal0l/mdeberta-v3-base-squad2',
   'threshold': 0.5},
  'triples_extraction': {'stage': 'triples_extraction',
   'batch_size': 16,
   'buffer_size': 16,
   'type': 'two-step-llm/deepseek-v4-flash'},
  'entities_normalization': {'stage': 'entities_normalization',
   'batch_size': 128,
   'buffer_size': 128},
  'rule_based_predicate_cleaning': {